# Classification Models: Logistic Regression and k-Nearest Neighbors

---

In this notebook we tackle one of the most common tasks in data science: **binary classification**. Given a set of patient measurements, can we predict whether someone is at risk for heart disease?

We will work with a simulated dataset inspired by the [UCI Heart Disease dataset](https://archive.ics.uci.edu/ml/datasets/heart+disease), covering roughly 500 patients with clinical features like cholesterol, blood pressure, and exercise habits.

### What You'll Learn

1. **Data simulation and exploration** — creating a realistic classification dataset and understanding its structure
2. **Preprocessing** — handling missing values, feature scaling with `StandardScaler`, and stratified train-test splits
3. **Logistic Regression** — fitting, predicting, and interpreting coefficients
4. **k-Nearest Neighbors (kNN)** — understanding the role of k and the importance of scaling
5. **LASSO (L1) regularization** — feature selection through coefficient shrinkage
6. **Polynomial features** — capturing non-linear decision boundaries
7. **Model evaluation** — confusion matrices, precision, recall, F1, cross-validation, ROC curves, and AUC

### Prerequisites

- Comfort with Python, NumPy, and Pandas (see Notebooks 01–02)
- A working environment with `scikit-learn`, `matplotlib`, `numpy`, and `pandas`

---

## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    precision_score, recall_score, f1_score, roc_curve, auc
)

warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 12

---

## 2. Simulating the Heart Disease Dataset

Real clinical datasets are often protected by privacy regulations. Instead, we will simulate a dataset that mirrors the statistical properties of the UCI Heart Disease data. This gives us full control over the ground truth and lets us focus on methodology.

Our dataset contains **500 patients** with the following features:

| Feature | Description |
|---|---|
| `age` | Patient age in years |
| `cholesterol` | Serum cholesterol in mg/dL |
| `blood_pressure` | Resting systolic blood pressure in mmHg |
| `max_heart_rate` | Maximum heart rate achieved during exercise |
| `exercise_hours_per_week` | Hours of exercise per week |
| `bmi` | Body mass index |
| `fasting_blood_sugar` | Fasting blood sugar in mg/dL |
| `chest_pain_type` | Categorical: 0 = typical angina, 1 = atypical angina, 2 = non-anginal, 3 = asymptomatic |
| **`heart_disease`** | **Target: 1 = disease present, 0 = no disease** |

In [ ]:
np.random.seed(42)
n = 500

# --- Generate features ---
age = np.random.normal(54, 9, n).clip(29, 80).round(0)
cholesterol = np.random.normal(240, 45, n).clip(120, 400).round(0)
blood_pressure = np.random.normal(130, 17, n).clip(90, 200).round(0)
max_heart_rate = np.random.normal(150, 23, n).clip(70, 210).round(0)
exercise_hours = np.random.exponential(3, n).clip(0, 20).round(1)
bmi = np.random.normal(27.5, 5, n).clip(16, 48).round(1)
fasting_blood_sugar = np.random.normal(105, 25, n).clip(60, 250).round(0)
chest_pain_type = np.random.choice([0, 1, 2, 3], n, p=[0.25, 0.25, 0.30, 0.20])

# --- Build a realistic target variable ---
# The log-odds of heart disease increase with age, cholesterol, blood pressure, BMI,
# and fasting blood sugar, and decrease with exercise and max heart rate.
logit = (
    -6.0
    + 0.05 * (age - 54)
    + 0.008 * (cholesterol - 240)
    + 0.02 * (blood_pressure - 130)
    - 0.015 * (max_heart_rate - 150)
    - 0.12 * (exercise_hours - 3)
    + 0.04 * (bmi - 27.5)
    + 0.005 * (fasting_blood_sugar - 105)
    + 0.3 * (chest_pain_type == 0).astype(float)
    + 0.6 * (chest_pain_type == 3).astype(float)
    + np.random.normal(0, 0.5, n)  # noise
)

prob = 1 / (1 + np.exp(-logit))
heart_disease = (np.random.rand(n) < prob).astype(int)

# --- Assemble DataFrame ---
df = pd.DataFrame({
    'age': age,
    'cholesterol': cholesterol,
    'blood_pressure': blood_pressure,
    'max_heart_rate': max_heart_rate,
    'exercise_hours_per_week': exercise_hours,
    'bmi': bmi,
    'fasting_blood_sugar': fasting_blood_sugar,
    'chest_pain_type': chest_pain_type,
    'heart_disease': heart_disease
})

# --- Inject a few missing values to practice handling them ---
rng = np.random.default_rng(99)
for col in ['cholesterol', 'bmi', 'fasting_blood_sugar']:
    missing_idx = rng.choice(n, size=8, replace=False)
    df.loc[missing_idx, col] = np.nan

print(f"Dataset shape: {df.shape}")
df.head(10)

---

## 3. Exploratory Data Analysis

Before building any model, we need to understand our data. Key questions:
- How are the classes distributed? Is the dataset balanced?
- Are there missing values?
- How do features relate to the target?

In [ ]:
# Basic info
print("=== Data Types and Missing Values ===")
print(df.info())
print("\n=== Missing value counts ===")
print(df.isnull().sum())
print(f"\nTotal rows with any missing value: {df.isnull().any(axis=1).sum()}")

In [ ]:
# Class balance
class_counts = df['heart_disease'].value_counts()
print("=== Class Balance ===")
print(class_counts)
print(f"\nDisease prevalence: {class_counts[1] / len(df):.1%}")

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['No Disease (0)', 'Disease (1)'], class_counts.values,
       color=['#5a9bd5', '#e05555'], edgecolor='white')
ax.set_ylabel('Count')
ax.set_title('Heart Disease Class Distribution')
for i, v in enumerate(class_counts.values):
    ax.text(i, v + 5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics by class
print("=== Feature Means by Heart Disease Status ===")
df.groupby('heart_disease').mean(numeric_only=True).round(2).T

In [ ]:
# Visualize feature distributions by class
continuous_features = ['age', 'cholesterol', 'blood_pressure', 'max_heart_rate',
                       'exercise_hours_per_week', 'bmi', 'fasting_blood_sugar']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

for i, feat in enumerate(continuous_features):
    ax = axes[i]
    for label, color in [(0, '#5a9bd5'), (1, '#e05555')]:
        subset = df[df['heart_disease'] == label][feat].dropna()
        ax.hist(subset, bins=20, alpha=0.6, color=color, edgecolor='white',
                label=f'{label}', density=True)
    ax.set_title(feat, fontsize=11)
    ax.legend(title='disease', fontsize=8)

# Hide the unused subplot
axes[-1].set_visible(False)

fig.suptitle('Feature Distributions by Heart Disease Status', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Some patterns are already visible: patients with heart disease tend to be older, have higher cholesterol and blood pressure, lower max heart rate, less exercise, and higher BMI. These are exactly the kinds of signals a classification model can learn.

---

## 4. Preprocessing

Before training models we need to:
1. **Handle missing values** — fill them with median values (a robust choice for clinical data)
2. **Split the data** into training and test sets with **stratification** (preserving class proportions)
3. **Scale features** — critical for kNN and helpful for logistic regression with regularization

### 4.1 Handling Missing Values

We saw a small number of missing values in `cholesterol`, `bmi`, and `fasting_blood_sugar`. In practice, you would investigate *why* values are missing (random? systematic?). Here we will use **median imputation** — replacing missing values with the column median, which is robust to outliers.

In [ ]:
# Impute missing values with column medians
print("Missing values BEFORE imputation:")
print(df.isnull().sum()[df.isnull().sum() > 0])

df_clean = df.copy()
for col in ['cholesterol', 'bmi', 'fasting_blood_sugar']:
    median_val = df_clean[col].median()
    df_clean[col].fillna(median_val, inplace=True)
    print(f"  Filled {col} missing values with median = {median_val}")

print(f"\nMissing values AFTER imputation: {df_clean.isnull().sum().sum()}")

### 4.2 Train-Test Split

We hold out 20% of the data for testing. The `stratify` parameter ensures that both the training set and the test set have the same proportion of disease-positive patients as the full dataset.

In [ ]:
feature_cols = ['age', 'cholesterol', 'blood_pressure', 'max_heart_rate',
                'exercise_hours_per_week', 'bmi', 'fasting_blood_sugar', 'chest_pain_type']

X = df_clean[feature_cols].values
y = df_clean['heart_disease'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"\nClass proportions (train): {np.mean(y_train):.3f}")
print(f"Class proportions (test):  {np.mean(y_test):.3f}")

### 4.3 Feature Scaling

**Why scale?** Features like `cholesterol` (range ~120–400) and `exercise_hours_per_week` (range ~0–20) live on very different scales. Algorithms that rely on distances (kNN) or gradient-based optimization (logistic regression with regularization) are sensitive to this.

`StandardScaler` transforms each feature to have **mean 0** and **standard deviation 1**.

**Important:** We fit the scaler on the training data only, then transform both train and test. This prevents **data leakage** — the test set should have no influence on preprocessing.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("After scaling (training set):")
print(f"  Means:  {X_train_scaled.mean(axis=0).round(4)}")
print(f"  Stds:   {X_train_scaled.std(axis=0).round(4)}")

---

## 5. Logistic Regression

Logistic regression is the workhorse of binary classification. Despite the name, it is a **classification** algorithm. It models the probability that an observation belongs to the positive class using the **logistic (sigmoid) function**:

$$P(y=1 | \mathbf{x}) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p)}}$$

The decision boundary is the set of points where this probability equals 0.5. On one side, we predict class 1; on the other, class 0.

In [ ]:
# Fit an unregularized logistic regression
# Setting penalty='none' and a large max_iter to ensure convergence
lr_model = LogisticRegression(penalty=None, max_iter=5000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

# Predictions on test set
y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

print("=== Logistic Regression (no regularization) ===")
print(f"Training accuracy: {lr_model.score(X_train_scaled, y_train):.4f}")
print(f"Test accuracy:     {accuracy_score(y_test, y_pred_lr):.4f}")

### 5.1 Evaluating with a Confusion Matrix

Accuracy alone can be misleading, especially with imbalanced classes. The **confusion matrix** breaks predictions into four categories:

| | Predicted 0 | Predicted 1 |
|---|---|---|
| **Actual 0** | True Negative (TN) | False Positive (FP) |
| **Actual 1** | False Negative (FN) | True Positive (TP) |

From these, we compute:
- **Precision** = TP / (TP + FP) — "Of those we predicted positive, how many truly are?"
- **Recall** = TP / (TP + FN) — "Of all actual positives, how many did we catch?"
- **F1-score** = harmonic mean of precision and recall

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred_lr)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['No Disease', 'Disease'])
ax.set_yticklabels(['No Disease', 'Disease'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix \u2014 Logistic Regression')

# Annotate cells
for i in range(2):
    for j in range(2):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                fontsize=18, fontweight='bold', color=color)

plt.tight_layout()
plt.show()

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred_lr, target_names=['No Disease', 'Disease']))

### 5.2 Metric Breakdown

Let's compute each metric individually to reinforce what they mean.

In [ ]:
acc = accuracy_score(y_test, y_pred_lr)
prec = precision_score(y_test, y_pred_lr)
rec = recall_score(y_test, y_pred_lr)
f1 = f1_score(y_test, y_pred_lr)

print(f"Accuracy:  {acc:.4f}  -- Fraction of all predictions that are correct")
print(f"Precision: {prec:.4f}  -- Of predicted disease cases, how many truly have it")
print(f"Recall:    {rec:.4f}  -- Of actual disease cases, how many we detected")
print(f"F1-score:  {f1:.4f}  -- Balance between precision and recall")
print()
print("In a medical setting, recall is often prioritized: missing a sick patient")
print("(false negative) can be far worse than a false alarm (false positive).")

---

## 6. Interpreting Logistic Regression Coefficients

One major advantage of logistic regression is **interpretability**. Each coefficient tells us how a one-standard-deviation increase in that feature changes the log-odds of heart disease (since we scaled the features).

- **Positive coefficient** → higher feature value increases the probability of disease
- **Negative coefficient** → higher feature value decreases the probability of disease

In [ ]:
coef_df = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': lr_model.coef_[0]
}).sort_values('coefficient', ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#e05555' if c > 0 else '#5a9bd5' for c in coef_df['coefficient']]
ax.barh(coef_df['feature'], coef_df['coefficient'], color=colors, edgecolor='white')
ax.set_xlabel('Coefficient (standardized)')
ax.set_title('Logistic Regression Coefficients')
ax.axvline(x=0, color='gray', linewidth=0.8, linestyle='--')
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("Red bars (positive) = increases risk of heart disease")
print("Blue bars (negative) = decreases risk of heart disease")
print("\nLargest risk factors appear to be those with the largest absolute coefficients.")
print(coef_df.to_string(index=False))

---

## 7. k-Nearest Neighbors (kNN) Classification

kNN is a fundamentally different approach: instead of learning a parametric decision boundary, it classifies each new point by looking at the **k closest training examples** and taking a majority vote.

Key considerations:
- **Feature scaling is essential** — kNN uses distances, so unscaled features with large ranges dominate
- **Choice of k** matters — small k is flexible but noisy; large k is smooth but may miss local structure
- kNN does not produce coefficients or a simple equation — it is a "lazy learner"

In [ ]:
# Fit kNN with k=5 (a common default)
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)

y_pred_knn = knn_model.predict(X_test_scaled)
y_prob_knn = knn_model.predict_proba(X_test_scaled)[:, 1]

print("=== kNN Classification (k=5) ===")
print(f"Training accuracy: {knn_model.score(X_train_scaled, y_train):.4f}")
print(f"Test accuracy:     {accuracy_score(y_test, y_pred_knn):.4f}")
print()
print(classification_report(y_test, y_pred_knn, target_names=['No Disease', 'Disease']))

### 7.1 How Does k Affect Performance?

Let's systematically try different values of k and track how training and test accuracy change. This is a classic illustration of the **bias-variance tradeoff**:

- **Small k** (e.g., k=1): Very flexible, potentially overfitting — training accuracy near 100%, but test accuracy may suffer
- **Large k** (e.g., k=50): Very smooth, potentially underfitting — may miss important local patterns

In [ ]:
k_values = list(range(1, 52, 2))  # odd values to avoid ties
train_accs = []
test_accs = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    train_accs.append(knn.score(X_train_scaled, y_train))
    test_accs.append(knn.score(X_test_scaled, y_test))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(k_values, train_accs, 'o-', color='#e05555', markersize=4, label='Training accuracy')
ax.plot(k_values, test_accs, 's-', color='#5a9bd5', markersize=4, label='Test accuracy')
ax.set_xlabel('k (number of neighbors)')
ax.set_ylabel('Accuracy')
ax.set_title('kNN: Training vs. Test Accuracy for Different k')
ax.legend()
ax.set_xticks(k_values[::2])
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best_k = k_values[np.argmax(test_accs)]
print(f"Best test accuracy: {max(test_accs):.4f} at k={best_k}")

Notice how at k=1 the training accuracy is perfect (each point is its own nearest neighbor!) but the test accuracy is lower. As k increases, training accuracy drops but test accuracy generally stabilizes or improves up to a point, then may decrease if k becomes too large.

### 7.2 Cross-Validation for Selecting k

Rather than relying on a single train-test split to pick k, we can use **cross-validation**. This gives us a more robust estimate of how each k will perform on unseen data.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_means = []
cv_stds = []
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=cv, scoring='accuracy')
    cv_means.append(scores.mean())
    cv_stds.append(scores.std())

cv_means = np.array(cv_means)
cv_stds = np.array(cv_stds)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(k_values, cv_means, 'o-', color='#2c7bb6', markersize=5)
ax.fill_between(k_values, cv_means - cv_stds, cv_means + cv_stds, alpha=0.2, color='#2c7bb6')
ax.set_xlabel('k (number of neighbors)')
ax.set_ylabel('Cross-validated accuracy')
ax.set_title('5-Fold Cross-Validation Accuracy for kNN')
ax.set_xticks(k_values[::2])
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best_k_cv = k_values[np.argmax(cv_means)]
print(f"Best CV accuracy: {cv_means.max():.4f} +/- {cv_stds[np.argmax(cv_means)]:.4f} at k={best_k_cv}")

In [ ]:
# Refit kNN with the best k from cross-validation and evaluate on test set
knn_best = KNeighborsClassifier(n_neighbors=best_k_cv)
knn_best.fit(X_train_scaled, y_train)
y_pred_knn_best = knn_best.predict(X_test_scaled)
y_prob_knn_best = knn_best.predict_proba(X_test_scaled)[:, 1]

print(f"=== kNN with k={best_k_cv} (selected via CV) ===")
print(f"Test accuracy: {accuracy_score(y_test, y_pred_knn_best):.4f}")
print()
print(classification_report(y_test, y_pred_knn_best, target_names=['No Disease', 'Disease']))

---

## 8. Comparing Logistic Regression vs. kNN

Let's put the two models side by side. We will also cross-validate logistic regression for a fair comparison.

In [ ]:
# Cross-validate logistic regression
lr_cv_scores = cross_val_score(
    LogisticRegression(penalty=None, max_iter=5000, random_state=42),
    X_train_scaled, y_train, cv=cv, scoring='accuracy'
)

knn_cv_scores = cross_val_score(
    KNeighborsClassifier(n_neighbors=best_k_cv),
    X_train_scaled, y_train, cv=cv, scoring='accuracy'
)

print("=== 5-Fold Cross-Validation Comparison ===")
print(f"Logistic Regression:  {lr_cv_scores.mean():.4f} +/- {lr_cv_scores.std():.4f}")
print(f"kNN (k={best_k_cv}):            {knn_cv_scores.mean():.4f} +/- {knn_cv_scores.std():.4f}")

# Side-by-side confusion matrices on test set
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, y_pred, title in zip(axes,
                              [y_pred_lr, y_pred_knn_best],
                              ['Logistic Regression', f'kNN (k={best_k_cv})']):
    cm = confusion_matrix(y_test, y_pred)
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['No Disease', 'Disease'])
    ax.set_yticklabels(['No Disease', 'Disease'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(title)
    for i in range(2):
        for j in range(2):
            color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    fontsize=16, fontweight='bold', color=color)

plt.tight_layout()
plt.show()

---

## 9. LASSO Logistic Regression for Feature Selection

Standard logistic regression uses all features. But what if some features are irrelevant or redundant? **LASSO (L1) regularization** adds a penalty proportional to the absolute value of the coefficients:

$$\text{minimize} \quad -\text{log-likelihood} + C^{-1} \sum_{j=1}^{p} |\beta_j|$$

This penalty pushes some coefficients all the way to zero, effectively **selecting** the most important features. The parameter `C` controls the regularization strength: smaller C means stronger regularization (more coefficients shrunk to zero).

In [ ]:
# Try several values of C
C_values = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]

results = []
for C in C_values:
    lasso_lr = LogisticRegression(penalty='l1', C=C, solver='liblinear',
                                  max_iter=5000, random_state=42)
    scores = cross_val_score(lasso_lr, X_train_scaled, y_train, cv=cv, scoring='accuracy')
    lasso_lr.fit(X_train_scaled, y_train)
    n_nonzero = np.sum(lasso_lr.coef_[0] != 0)
    results.append({
        'C': C,
        'cv_accuracy': scores.mean(),
        'cv_std': scores.std(),
        'n_features': n_nonzero
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
# Visualize how coefficients change with regularization strength
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: CV accuracy vs C
ax = axes[0]
ax.semilogx(results_df['C'], results_df['cv_accuracy'], 'o-', color='#2c7bb6')
ax.fill_between(results_df['C'],
                results_df['cv_accuracy'] - results_df['cv_std'],
                results_df['cv_accuracy'] + results_df['cv_std'],
                alpha=0.2, color='#2c7bb6')
ax.set_xlabel('C (inverse regularization strength)')
ax.set_ylabel('Cross-validated accuracy')
ax.set_title('LASSO LR: Accuracy vs. C')
ax.grid(alpha=0.3)

# Right: Coefficient paths
ax = axes[1]
coef_paths = []
for C in C_values:
    lasso_lr = LogisticRegression(penalty='l1', C=C, solver='liblinear',
                                  max_iter=5000, random_state=42)
    lasso_lr.fit(X_train_scaled, y_train)
    coef_paths.append(lasso_lr.coef_[0])

coef_paths = np.array(coef_paths)
for j, feat in enumerate(feature_cols):
    ax.semilogx(C_values, coef_paths[:, j], 'o-', markersize=4, label=feat)

ax.set_xlabel('C (inverse regularization strength)')
ax.set_ylabel('Coefficient value')
ax.set_title('LASSO Coefficient Paths')
ax.legend(fontsize=8, loc='best')
ax.axhline(y=0, color='gray', linewidth=0.8, linestyle='--')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nAt strong regularization (small C), only the most important features survive.")
print("As C grows, regularization weakens and all coefficients approach their unpenalized values.")

In [ ]:
# Select best C and show final LASSO model
best_C = results_df.loc[results_df['cv_accuracy'].idxmax(), 'C']
print(f"Best C from cross-validation: {best_C}")

lasso_final = LogisticRegression(penalty='l1', C=best_C, solver='liblinear',
                                  max_iter=5000, random_state=42)
lasso_final.fit(X_train_scaled, y_train)
y_pred_lasso = lasso_final.predict(X_test_scaled)
y_prob_lasso = lasso_final.predict_proba(X_test_scaled)[:, 1]

print(f"Test accuracy: {accuracy_score(y_test, y_pred_lasso):.4f}")
print()

# Feature importance from LASSO
lasso_coefs = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': lasso_final.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)

print("=== LASSO Feature Importance (by |coefficient|) ===")
for _, row in lasso_coefs.iterrows():
    marker = '***' if row['coefficient'] != 0 else '   (dropped)'
    print(f"  {row['feature']:>28s}: {row['coefficient']:+.4f}  {marker}")

---

## 9.5 Polynomial Features for Non-Linear Boundaries

Standard logistic regression finds a **linear** decision boundary. But what if the true boundary between healthy and sick patients is curved? One approach is to create **polynomial features** — products and powers of existing features — and let logistic regression use them.

We will add degree-2 polynomial features and see if this improves performance.

In [ ]:
# Create polynomial features (degree 2, interaction terms + squared terms)
poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)

print(f"Original features: {X_train_scaled.shape[1]}")
print(f"After polynomial expansion: {X_train_poly.shape[1]}")

# Fit logistic regression with polynomial features (use L1 to handle the many features)
lr_poly = LogisticRegression(penalty='l1', C=1.0, solver='liblinear',
                              max_iter=5000, random_state=42)
lr_poly.fit(X_train_poly, y_train)

y_pred_poly = lr_poly.predict(X_test_poly)
y_prob_poly = lr_poly.predict_proba(X_test_poly)[:, 1]

# Cross-validate
poly_cv_scores = cross_val_score(
    LogisticRegression(penalty='l1', C=1.0, solver='liblinear',
                       max_iter=5000, random_state=42),
    X_train_poly, y_train, cv=cv, scoring='accuracy'
)

print(f"\nPolynomial LR (degree 2):")
print(f"  CV accuracy:   {poly_cv_scores.mean():.4f} +/- {poly_cv_scores.std():.4f}")
print(f"  Test accuracy: {accuracy_score(y_test, y_pred_poly):.4f}")
print(f"  Non-zero coefficients: {np.sum(lr_poly.coef_[0] != 0)} / {X_train_poly.shape[1]}")
print()
print("Adding polynomial features can capture non-linear relationships,")
print("but the improvement depends on whether such non-linearity exists in the data.")

---

## 10. ROC Curves and AUC

The **Receiver Operating Characteristic (ROC) curve** shows the tradeoff between the **True Positive Rate** (recall) and the **False Positive Rate** as we vary the classification threshold.

- A perfect classifier hugs the top-left corner (TPR=1, FPR=0)
- A random classifier follows the diagonal (AUC = 0.5)
- **AUC** (Area Under the Curve) summarizes overall discriminative ability: higher is better

ROC curves are particularly useful because they are **threshold-independent** — they show performance across all possible decision thresholds, not just the default 0.5.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

# Plot ROC curves for all models
models_for_roc = [
    ('Logistic Regression', y_prob_lr, '#2c7bb6'),
    (f'kNN (k={best_k_cv})', y_prob_knn_best, '#d7191c'),
    ('LASSO LR', y_prob_lasso, '#1a9641'),
    ('Polynomial LR (deg 2)', y_prob_poly, '#fdae61'),
]

for name, y_prob, color in models_for_roc:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC = {roc_auc:.3f})')

# Diagonal reference line
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random (AUC = 0.500)')

ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title('ROC Curves \u2014 Model Comparison')
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
ax.grid(alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

In [ ]:
# Summary table
summary_rows = []
model_preds = [
    ('Logistic Regression', y_pred_lr, y_prob_lr),
    (f'kNN (k={best_k_cv})', y_pred_knn_best, y_prob_knn_best),
    ('LASSO LR', y_pred_lasso, y_prob_lasso),
    ('Polynomial LR', y_pred_poly, y_prob_poly),
]

for name, y_pred, y_prob in model_preds:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    summary_rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'AUC': auc(fpr, tpr)
    })

summary_df = pd.DataFrame(summary_rows).set_index('Model')
print("=== Model Comparison Summary ===")
print(summary_df.round(4).to_string())

---

## 11. Wrap-Up

In this notebook we built a complete binary classification pipeline from scratch:

- **Data creation and EDA** — We simulated a realistic heart disease dataset and explored feature distributions and class balance.

- **Preprocessing** — We handled missing values with median imputation, split the data with stratification, and scaled features with `StandardScaler` (fitting only on training data to prevent leakage).

- **Logistic Regression** — A parametric model that estimates class probabilities via the sigmoid function. Its coefficients are directly interpretable: positive values increase disease risk, negative values decrease it.

- **k-Nearest Neighbors** — A non-parametric, distance-based approach. We saw how the choice of k governs the bias-variance tradeoff and used cross-validation to select the best k.

- **LASSO regularization** — L1 penalty shrinks some coefficients to exactly zero, performing automatic feature selection. We visualized coefficient paths as regularization strength varied.

- **Polynomial features** — Expanding the feature space to capture non-linear decision boundaries, at the cost of more parameters.

- **Evaluation** — We went beyond accuracy to examine precision, recall, F1-score, confusion matrices, and ROC/AUC — each capturing a different aspect of classifier quality.

### Key Takeaways

1. **Always look at more than accuracy.** In medical contexts, recall (catching all true positives) is often more important than precision.
2. **Feature scaling matters** for distance-based methods (kNN) and regularized models (LASSO).
3. **Cross-validation** gives a more reliable estimate of model performance than a single train-test split.
4. **Regularization** helps prevent overfitting and can reveal which features matter most.
5. **ROC curves** let you compare models across all possible decision thresholds, not just the default 0.5.

Next up: decision trees, random forests, and ensemble methods.